# Pruebas de hipótesis: precios de WTI y Brent

Este notebook concentra las cuatro comparaciones solicitadas sin modificar los análisis existentes:

1. WTI vs Brent en el período 1.
2. WTI vs Brent en el período 2.
3. WTI del período 1 vs WTI del período 2.
4. Brent del período 1 vs Brent del período 2.

Se usa el CSV local del proyecto. El **período 1** abarca desde el 01/01/2024 hasta el 28/02/2026 y el **período 2** desde el 01/04/2026 hasta la última fecha disponible. Se excluyen todos los registros de **marzo de 2026**, de acuerdo con el criterio definido para el proyecto.

## Hipótesis y nivel de significación

Se adopta $\alpha = 0.05$ y pruebas bilaterales.

- **Levene**: $H_0$ establece que las varianzas son iguales.
- **t de Student o Welch**: $H_0$ establece que las medias son iguales. Si Levene rechaza la igualdad de varianzas, se usa Welch (`equal_var=False`).
- **Mann–Whitney U**: $H_0$ establece que ambas muestras tienen la misma distribución. Cuando sus formas son comparables, puede interpretarse como una comparación de ubicación o mediana.

La decisión se expresa como **rechazar** o **no rechazar** $H_0$; un valor p alto no demuestra que $H_0$ sea verdadera.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
from IPython.display import Markdown, display

ALFA = 0.05
pd.set_option("display.max_colwidth", 120)

## 1. Carga y validación del CSV

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

archivos_csv = list((PROJECT_ROOT / "data").glob("eia_precios_crudo_diarios_*.csv"))
if not archivos_csv:
    raise FileNotFoundError("No se encontró el CSV de precios diarios dentro de data/.")

CSV_PATH = max(archivos_csv, key=lambda ruta: ruta.stat().st_mtime)
df = pd.read_csv(CSV_PATH)

columnas_requeridas = {"period", "series", "value"}
faltantes = columnas_requeridas.difference(df.columns)
if faltantes:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {sorted(faltantes)}")

df = df.copy()
df["period"] = pd.to_datetime(df["period"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.loc[df["series"].isin(["RWTC", "RBRTE"])].copy()

if df[["period", "series", "value"]].isna().any().any():
    raise ValueError("El CSV contiene fechas, series o precios inválidos.")
if df.duplicated(["period", "series"]).any():
    raise ValueError("El CSV contiene más de un precio por fecha y serie.")
if not np.isfinite(df["value"]).all():
    raise ValueError("El CSV contiene precios no finitos.")

print(f"Archivo utilizado: {CSV_PATH.name}")
print(f"Registros WTI/Brent validados: {len(df):,}")
print(f"Cobertura: {df['period'].min():%d/%m/%Y} a {df['period'].max():%d/%m/%Y}")

Archivo utilizado: eia_precios_crudo_diarios_2024-01_a_2026-09.csv
Registros WTI/Brent validados: 1,368
Cobertura: 02/01/2024 a 22/09/2026


## 2. Definición de períodos y exclusión de marzo de 2026

In [3]:
INICIO_PERIODO_1 = pd.Timestamp("2024-01-01")
FIN_PERIODO_1 = pd.Timestamp("2026-02-28")
INICIO_PERIODO_2 = pd.Timestamp("2026-04-01")
FIN_PERIODO_2 = df["period"].max()

es_marzo_2026 = df["period"].between("2026-03-01", "2026-03-31")
registros_marzo_descartados = int(es_marzo_2026.sum())
datos_sin_marzo_2026 = df.loc[~es_marzo_2026].copy()

periodo_1 = datos_sin_marzo_2026.loc[
    datos_sin_marzo_2026["period"].between(INICIO_PERIODO_1, FIN_PERIODO_1)
].copy()
periodo_1["periodo_analisis"] = "Período 1"

periodo_2 = datos_sin_marzo_2026.loc[
    datos_sin_marzo_2026["period"].between(INICIO_PERIODO_2, FIN_PERIODO_2)
].copy()
periodo_2["periodo_analisis"] = "Período 2"

datos_analisis = pd.concat([periodo_1, periodo_2], ignore_index=True)
datos_analisis["crudo"] = datos_analisis["series"].map({"RWTC": "WTI", "RBRTE": "Brent"})

if datos_analisis.empty:
    raise ValueError("No quedaron datos dentro de los períodos definidos.")
if datos_analisis["period"].between("2026-03-01", "2026-03-31").any():
    raise AssertionError("La exclusión de marzo de 2026 no se aplicó correctamente.")

cobertura = (
    datos_analisis.groupby(["periodo_analisis", "crudo"], observed=True)
    .agg(desde=("period", "min"), hasta=("period", "max"), observaciones=("value", "size"))
    .reset_index()
)
print(f"Registros descartados de marzo de 2026: {registros_marzo_descartados}")
display(cobertura)

Registros descartados de marzo de 2026: 44


,periodo_analisis,crudo,desde,hasta,observaciones
0,Período 1,Brent,2024-01-02,2026-02-27,547
1,Período 1,WTI,2024-01-02,2026-02-27,537
2,Período 2,Brent,2026-04-01,2026-09-22,120
3,Período 2,WTI,2026-04-01,2026-09-22,120


## 3. Resumen descriptivo de las muestras

In [4]:
resumen_descriptivo = (
    datos_analisis.groupby(["periodo_analisis", "crudo"], observed=True)["value"]
    .agg(observaciones="count", media="mean", mediana="median", varianza="var", desvio_estandar="std", minimo="min", maximo="max")
    .reset_index()
)
display(resumen_descriptivo.round(3))

,periodo_analisis,crudo,observaciones,media,mediana,varianza,desvio_estandar,minimo,maximo
0,Período 1,Brent,547,74.340,73.780,59.745,7.729,59.93,93.12
1,Período 1,WTI,537,70.368,69.780,61.411,7.837,55.44,87.69
2,Período 2,Brent,120,98.462,97.030,261.783,16.180,68.53,138.21
3,Período 2,WTI,120,91.154,92.025,127.989,11.313,69.60,114.58


## 4. Función para ejecutar las tres pruebas

La prueba t se adapta al resultado de Levene. Si las varianzas no pueden considerarse iguales, se aplica la versión de Welch.

In [5]:
def ejecutar_pruebas(nombre, etiqueta_a, grupo_a, etiqueta_b, grupo_b, alfa=ALFA):
    a = pd.Series(grupo_a, dtype="float64").dropna()
    b = pd.Series(grupo_b, dtype="float64").dropna()

    if len(a) < 2 or len(b) < 2:
        raise ValueError(f"{nombre}: se necesitan al menos dos observaciones por grupo.")

    estadistico_levene, p_levene = stats.levene(a, b, center="median")
    varianzas_iguales = bool(p_levene >= alfa)
    estadistico_t, p_t = stats.ttest_ind(
        a, b, equal_var=varianzas_iguales, alternative="two-sided"
    )
    estadistico_u, p_u = stats.mannwhitneyu(
        a, b, alternative="two-sided", method="auto"
    )

    prueba_media = "t de Student" if varianzas_iguales else "t de Welch"
    descriptivos = {
        "comparacion": nombre,
        "grupo_a": etiqueta_a,
        "grupo_b": etiqueta_b,
        "n_a": len(a),
        "n_b": len(b),
        "media_a": a.mean(),
        "media_b": b.mean(),
        "mediana_a": a.median(),
        "mediana_b": b.median(),
    }

    pruebas = [
        {
            "comparacion": nombre,
            "prueba": "Levene",
            "hipotesis_nula": "Las varianzas son iguales",
            "estadistico": estadistico_levene,
            "p_valor": p_levene,
            "decision": "Rechazar H0" if p_levene < alfa else "No rechazar H0",
        },
        {
            "comparacion": nombre,
            "prueba": prueba_media,
            "hipotesis_nula": "Las medias son iguales",
            "estadistico": estadistico_t,
            "p_valor": p_t,
            "decision": "Rechazar H0" if p_t < alfa else "No rechazar H0",
        },
        {
            "comparacion": nombre,
            "prueba": "Mann–Whitney U",
            "hipotesis_nula": "Las distribuciones son iguales",
            "estadistico": estadistico_u,
            "p_valor": p_u,
            "decision": "Rechazar H0" if p_u < alfa else "No rechazar H0",
        },
    ]
    return descriptivos, pruebas

## 5. Ejecución de las cuatro comparaciones

In [6]:
def muestra(periodo, crudo):
    return datos_analisis.loc[
        datos_analisis["periodo_analisis"].eq(periodo)
        & datos_analisis["crudo"].eq(crudo),
        "value",
    ]

comparaciones = [
    ("WTI vs Brent - Período 1", "WTI P1", muestra("Período 1", "WTI"), "Brent P1", muestra("Período 1", "Brent")),
    ("WTI vs Brent - Período 2", "WTI P2", muestra("Período 2", "WTI"), "Brent P2", muestra("Período 2", "Brent")),
    ("WTI - Período 1 vs Período 2", "WTI P1", muestra("Período 1", "WTI"), "WTI P2", muestra("Período 2", "WTI")),
    ("Brent - Período 1 vs Período 2", "Brent P1", muestra("Período 1", "Brent"), "Brent P2", muestra("Período 2", "Brent")),
]

filas_descriptivas = []
filas_pruebas = []
for comparacion in comparaciones:
    descriptivos, pruebas = ejecutar_pruebas(*comparacion)
    filas_descriptivas.append(descriptivos)
    filas_pruebas.extend(pruebas)

resumen_comparaciones = pd.DataFrame(filas_descriptivas)
resultados_pruebas = pd.DataFrame(filas_pruebas)

display(resumen_comparaciones.round(3))
display(
    resultados_pruebas.assign(
        estadistico=resultados_pruebas["estadistico"].round(4),
        p_valor=resultados_pruebas["p_valor"].map(lambda valor: f"{valor:.6g}"),
    )
)

,comparacion,grupo_a,grupo_b,n_a,n_b,media_a,media_b,mediana_a,mediana_b
0,WTI vs Brent - Período 1,WTI P1,Brent P1,537,547,70.368,74.340,69.780,73.780
1,WTI vs Brent - Período 2,WTI P2,Brent P2,120,120,91.154,98.462,92.025,97.030
2,WTI - Período 1 vs Período 2,WTI P1,WTI P2,537,120,70.368,91.154,69.780,92.025
3,Brent - Período 1 vs Período 2,Brent P1,Brent P2,547,120,74.340,98.462,73.780,97.030


,comparacion,prueba,hipotesis_nula,estadistico,p_valor,decision
0,WTI vs Brent - Período 1,Levene,Las varianzas son iguales,0.2501,0.617088,No rechazar H0
1,WTI vs Brent - Período 1,t de Student,Las medias son iguales,-8.4009,1.37817e-16,Rechazar H0
2,WTI vs Brent - Período 1,Mann–Whitney U,Las distribuciones son iguales,106627.5000,5.78585e-15,Rechazar H0
3,WTI vs Brent - Período 2,Levene,Las varianzas son iguales,14.0642,0.000221914,Rechazar H0
4,WTI vs Brent - Período 2,t de Welch,Las medias son iguales,-4.0551,7.03063e-05,Rechazar H0
5,WTI vs Brent - Período 2,Mann–Whitney U,Las distribuciones son iguales,5277.0000,0.000350322,Rechazar H0
6,WTI - Período 1 vs Período 2,Levene,Las varianzas son iguales,34.6214,6.39134e-09,Rechazar H0
7,WTI - Período 1 vs Período 2,t de Welch,Las medias son iguales,-19.1273,1.50767e-41,Rechazar H0
8,WTI - Período 1 vs Período 2,Mann–Whitney U,Las distribuciones son iguales,4416.5000,1.6793e-49,Rechazar H0
9,Brent - Período 1 vs Período 2,Levene,Las varianzas son iguales,147.8216,7.4542e-31,Rechazar H0


## 6. Conclusiones automáticas

In [7]:
for fila in resumen_comparaciones.itertuples(index=False):
    pruebas = resultados_pruebas.loc[resultados_pruebas["comparacion"].eq(fila.comparacion)]
    levene = pruebas.loc[pruebas["prueba"].eq("Levene")].iloc[0]
    prueba_media = pruebas.loc[pruebas["prueba"].str.startswith("t de")].iloc[0]
    mann_whitney = pruebas.loc[pruebas["prueba"].eq("Mann–Whitney U")].iloc[0]

    conclusion_varianza = (
        "se rechaza la igualdad de varianzas"
        if levene.p_valor < ALFA
        else "no se rechaza la igualdad de varianzas"
    )
    conclusion_media = (
        "se observa una diferencia estadísticamente significativa entre las medias"
        if prueba_media.p_valor < ALFA
        else "no se observa evidencia suficiente de una diferencia entre las medias"
    )
    conclusion_distribucion = (
        "se observa una diferencia estadísticamente significativa entre las distribuciones"
        if mann_whitney.p_valor < ALFA
        else "no se observa evidencia suficiente de una diferencia entre las distribuciones"
    )

    texto = f"""### {fila.comparacion}

- **Muestras:** {fila.grupo_a} (n={fila.n_a}) y {fila.grupo_b} (n={fila.n_b}).
- **Medias:** {fila.media_a:.2f} y {fila.media_b:.2f} USD/barril.
- **Medianas:** {fila.mediana_a:.2f} y {fila.mediana_b:.2f} USD/barril.
- **Levene:** p={levene.p_valor:.6g}; {conclusion_varianza}.
- **{prueba_media.prueba}:** p={prueba_media.p_valor:.6g}; {conclusion_media}.
- **Mann–Whitney U:** p={mann_whitney.p_valor:.6g}; {conclusion_distribucion}.
"""
    display(Markdown(texto))

### WTI vs Brent - Período 1

- **Muestras:** WTI P1 (n=537) y Brent P1 (n=547).
- **Medias:** 70.37 y 74.34 USD/barril.
- **Medianas:** 69.78 y 73.78 USD/barril.
- **Levene:** p=0.617088; no se rechaza la igualdad de varianzas.
- **t de Student:** p=1.37817e-16; se observa una diferencia estadísticamente significativa entre las medias.
- **Mann–Whitney U:** p=5.78585e-15; se observa una diferencia estadísticamente significativa entre las distribuciones.


### WTI vs Brent - Período 2

- **Muestras:** WTI P2 (n=120) y Brent P2 (n=120).
- **Medias:** 91.15 y 98.46 USD/barril.
- **Medianas:** 92.03 y 97.03 USD/barril.
- **Levene:** p=0.000221914; se rechaza la igualdad de varianzas.
- **t de Welch:** p=7.03063e-05; se observa una diferencia estadísticamente significativa entre las medias.
- **Mann–Whitney U:** p=0.000350322; se observa una diferencia estadísticamente significativa entre las distribuciones.


### WTI - Período 1 vs Período 2

- **Muestras:** WTI P1 (n=537) y WTI P2 (n=120).
- **Medias:** 70.37 y 91.15 USD/barril.
- **Medianas:** 69.78 y 92.03 USD/barril.
- **Levene:** p=6.39134e-09; se rechaza la igualdad de varianzas.
- **t de Welch:** p=1.50767e-41; se observa una diferencia estadísticamente significativa entre las medias.
- **Mann–Whitney U:** p=1.6793e-49; se observa una diferencia estadísticamente significativa entre las distribuciones.


### Brent - Período 1 vs Período 2

- **Muestras:** Brent P1 (n=547) y Brent P2 (n=120).
- **Medias:** 74.34 y 98.46 USD/barril.
- **Medianas:** 73.78 y 97.03 USD/barril.
- **Levene:** p=7.4542e-31; se rechaza la igualdad de varianzas.
- **t de Welch:** p=1.78041e-32; se observa una diferencia estadísticamente significativa entre las medias.
- **Mann–Whitney U:** p=1.62249e-44; se observa una diferencia estadísticamente significativa entre las distribuciones.


## Consideración metodológica

Las pruebas se aplican a niveles diarios de precios, tal como fue solicitado. Los precios diarios forman series temporales y pueden presentar autocorrelación; por eso los valores p deben interpretarse como evidencia exploratoria. Para estudiar movimientos diarios con mayor independencia, una extensión natural consiste en repetir el análisis sobre retornos porcentuales.